# 02 – Feature-set trade-off: 20 features vs ~120, and do the ratios matter?

The app uses 15 hand-picked inputs plus 5 engineered features, so a person can fill in the form and the reasons stay readable. This notebook asks two questions, with confidence intervals:

1. What does the curated set cost against every column of `application_train.csv`?
2. Do the engineered ratios help? (ablation: the curated set without them)

All three models use the same LightGBM settings and the same train/validation applicants. Protected attributes (gender, age, family status) are excluded everywhere. **Validation only**: the test set stays untouched. Caveat: early stopping also uses validation, the same way for all three, so absolute AUCs are slightly optimistic but the comparison is like for like.

In [1]:
import lightgbm as lgb
import pandas as pd

from credit_risk.config import ID_COL, MODEL_INPUTS, TARGET
from credit_risk.data import find_train_file, load_splits
from credit_risk.evaluate import paired_bootstrap_auc
from credit_risk.models import RISK_DECREASES_WITH, fit_lgbm
from credit_risk.train import predict_default_proba

splits = load_splits()
y_val = splits.val[TARGET].to_numpy()
curated = fit_lgbm(splits.train, splits.val)
p_curated = predict_default_proba(curated, splits.val)

/Users/yoavdory/Desktop/Personal/Projects/glassbox/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Every raw column

In [2]:
raw = pd.read_csv(find_train_file()).set_index("SK_ID_CURR")
protected = ["CODE_GENDER", "DAYS_BIRTH", "NAME_FAMILY_STATUS"]
X_all = raw.drop(columns=["TARGET", *protected])
X_all["DAYS_EMPLOYED"] = X_all["DAYS_EMPLOYED"].mask(X_all["DAYS_EMPLOYED"] == 365243)
for col in X_all.select_dtypes(exclude="number").columns:
    X_all[col] = X_all[col].astype("category")

train_ids, val_ids = splits.train[ID_COL], splits.val[ID_COL]  # same applicants, same order
params = dict(n_estimators=3000, learning_rate=0.03, num_leaves=31, min_child_samples=200,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
              random_state=42, verbose=-1)
all_columns = lgb.LGBMClassifier(**params).fit(
    X_all.loc[train_ids], raw.loc[train_ids, "TARGET"],
    eval_X=X_all.loc[val_ids], eval_y=raw.loc[val_ids, "TARGET"],
    callbacks=[lgb.early_stopping(200, verbose=False)])
p_all = all_columns.predict_proba(X_all.loc[val_ids])[:, 1]
print(f"{X_all.shape[1]} columns")

117 columns


## 2. Curated set without the engineered ratios

In [3]:
RATIOS = ["payment_to_income", "credit_to_income", "loan_to_value", "credit_to_annuity"]
prepare = curated[:-1]  # the fitted feature and category steps
X_tr = prepare.transform(splits.train[MODEL_INPUTS]).drop(columns=RATIOS)
X_va = prepare.transform(splits.val[MODEL_INPUTS]).drop(columns=RATIOS)
no_ratios_params = {**curated[-1].get_params(),
                    "monotone_constraints": [-1 if c in RISK_DECREASES_WITH else 0 for c in X_tr.columns]}
no_ratios = lgb.LGBMClassifier(**no_ratios_params).fit(
    X_tr, splits.train[TARGET], eval_X=X_va, eval_y=splits.val[TARGET],
    callbacks=[lgb.early_stopping(200, verbose=False)])
p_no_ratios = no_ratios.predict_proba(X_va)[:, 1]

## Results with 95% paired-bootstrap intervals (validation, 1,000 resamples)

In [4]:
vs_all = paired_bootstrap_auc(y_val, p_all, p_curated)
vs_no_ratios = paired_bootstrap_auc(y_val, p_no_ratios, p_curated)

def show(name, s):
    return f"{name:<34} {s['estimate']:.4f}  [{s['low']:.4f}, {s['high']:.4f}]"

print(show('Curated (15 inputs + 5 engineered)', vs_all['b']))
print(show('All 117 raw columns', vs_all['a']))
print(show('Curated without the 4 ratios', vs_no_ratios['a']))
print()
print(show('Curated minus all-columns', vs_all['difference']))
print(show('Curated minus no-ratios', vs_no_ratios['difference']))

Curated (15 inputs + 5 engineered) 0.7584  [0.7513, 0.7654]
All 117 raw columns                0.7525  [0.7450, 0.7593]
Curated without the 4 ratios       0.7497  [0.7425, 0.7571]

Curated minus all-columns          0.0059  [0.0031, 0.0089]
Curated minus no-ratios            0.0087  [0.0068, 0.0106]


## Reading

| Model | Validation ROC-AUC [95% CI] |
|---|---|
| Curated: 15 inputs + 5 engineered | 0.758 [0.751, 0.765] |
| All 117 raw columns | 0.753 [0.745, 0.759] |
| Curated without the 4 ratios | 0.750 [0.743, 0.757] |

| Paired difference | Estimate [95% CI] |
|---|---|
| Curated minus all columns | +0.006 [+0.003, +0.009] |
| Curated minus no ratios | +0.009 [+0.007, +0.011] |

- **The curated set costs nothing here.** It is not worse than using every column; it is slightly better,
  by an amount that is small but outside the resampling noise.
- **The engineered ratios account for that edge.** Removing the four ratios lowers AUC by about 0.009,
  which puts the curated set back at roughly the all-columns level. A tree splits on one column at a time,
  so handing it a ratio directly is more efficient than letting it approximate one.

How far this goes, honestly:
- The intervals only cover sampling noise in the validation applicants. They say nothing about how much
  results would move if the models were retrained on a different training sample (one split, one seed).
- Early stopping used the same validation set for all three models, so absolute AUCs are slightly optimistic.
- The all-columns model got the same settings and no feature engineering of its own.
- Kaggle's best scores (~0.80) used six other tables: more *information*, not more columns of this one.

**Takeaway:** a small, readable feature set is a reasonable choice for this table; the differences are
real but small (under 0.01 AUC).